# Chapter 38: Bundle Adjustment

<a href="../lite/lab/index.html?path=ch38_bundle_adjustment.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Bundle Adjustment is the gold standard of visual reconstruction. It simultaneously
adjusts every camera pose and every 3D point so that all reprojection errors are
minimized. It is the most expensive step in visual SLAM, but also the one that
makes everything precise.

In this chapter we implement:
- **Reprojection error:** the fundamental residual in visual SLAM
- **Gauss Newton bundle adjustment:** jointly optimizing cameras and points
- **Local vs global BA:** the speed/accuracy tradeoff
- **Schur complement:** exploiting the sparse structure of the BA problem

```{admonition} What you will build
:class: tip

- Implement bundle adjustment: jointly optimize camera poses and 3D points using Gauss-Newton
- Compute reprojection errors and watch them decrease with each iteration
- Use the Schur complement trick to solve large BA problems efficiently
- Compare local BA (fast, recent frames only) with global BA (accurate, everything)

**Real world application:** Bundle adjustment is the gold standard of 3D reconstruction. It runs in every visual SLAM system, in photogrammetry software, and in Google Street View. After this chapter, you will have implemented the algorithm that makes precise 3D maps from camera images.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **Ceres Solver** | Google's BA solver, used in Cartographer and many reconstruction tools |
| **g2o** | BA solver used in ORB-SLAM's back end |
| **GTSAM** | Supports BA with Schur complement and incremental solving |
| **COLMAP** | Open source structure from motion with state of the art BA |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

### Shared Utilities

We define a simplified 2D camera model. Each camera has pose $(x, y, \theta)$
and observes 2D landmarks as **bearing angles**. This keeps the math clear while
preserving the essential structure of the BA problem.

In [ ]:
def rot2(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def project_bearing(cam_pose, landmark):
    """Project a 2D landmark to a bearing angle observation."""
    x, y, theta = cam_pose
    R = rot2(-theta)
    t = np.array([x, y])
    local = R @ (landmark - t)
    return np.arctan2(local[1], local[0])

def project_pixel(K, R, t, point_3d):
    """Project a 3D point to pixel coordinates. K is 3x3, R is 3x3, t is 3x1."""
    p_cam = R @ point_3d + t
    p_norm = p_cam[:2] / p_cam[2]
    pixel = K[:2, :2] @ p_norm + K[:2, 2]
    return pixel

## 38.1 Reprojection Error

The **reprojection error** for a single observation is the difference between
where a 3D point *actually* appears in the image and where our current estimates
of camera pose and point position *predict* it should appear:

$$\mathbf{e}_{ij} = \mathbf{u}_{ij} - \pi(T_i, \mathbf{p}_j)$$

where $\mathbf{u}_{ij}$ is the observed pixel location of point $j$ in camera $i$,
and $\pi(T_i, \mathbf{p}_j)$ is the predicted projection.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
focal_length = 500.0             # focal length (px)
cx_s1, cy_s1 = 320.0, 240.0     # principal point
pose_error = np.array([0.3, -0.2, 0.05])  # error in pose estimate
point_error = np.array([0.5, 0.3, -0.4])  # error in point estimate
# ─────────────────────────────────────────────────────────────────────────────

K = np.array([[focal_length, 0, cx_s1],
              [0, focal_length, cy_s1],
              [0, 0, 1.0]])

# True camera at origin, looking along z
R_true = np.eye(3)
t_true = np.zeros(3)

# True 3D point
pt_true = np.array([1.0, 0.5, 5.0])

# True observation (pixel)
obs_true = project_pixel(K, R_true, t_true, pt_true)

# Estimated camera (with error)
t_est = t_true + pose_error
pt_est = pt_true + point_error

# Predicted projection with wrong estimates
pred = project_pixel(K, R_true, t_est, pt_est)

# Reprojection error
reproj_err = obs_true - pred

print(f'True observation:       ({obs_true[0]:.1f}, {obs_true[1]:.1f}) px')
print(f'Predicted projection:   ({pred[0]:.1f}, {pred[1]:.1f}) px')
print(f'Reprojection error:     ({reproj_err[0]:.1f}, {reproj_err[1]:.1f}) px')
print(f'Error magnitude:        {np.linalg.norm(reproj_err):.1f} px')

In [ ]:
# Visualize multiple reprojection errors
np.random.seed(42)
n_pts_viz = 10
pts_true_viz = np.random.uniform(-2, 2, (n_pts_viz, 3))
pts_true_viz[:, 2] = np.random.uniform(3, 8, n_pts_viz)

# Noisy estimates
pts_est_viz = pts_true_viz + np.random.normal(0, 0.3, pts_true_viz.shape)
t_est_viz = np.random.normal(0, 0.1, 3)

fig, ax = plt.subplots(figsize=(8, 6))

for j in range(n_pts_viz):
    obs_j = project_pixel(K, R_true, t_true, pts_true_viz[j])
    pred_j = project_pixel(K, R_true, t_est_viz, pts_est_viz[j])
    err_j = np.linalg.norm(obs_j - pred_j)
    
    ax.scatter(obs_j[0], obs_j[1], c='forestgreen', s=60, zorder=5,
              edgecolors='k', linewidth=0.5)
    ax.scatter(pred_j[0], pred_j[1], c='tomato', s=40, marker='x', zorder=5)
    ax.annotate('', xy=pred_j, xytext=obs_j,
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.5))
    ax.annotate(f'{err_j:.1f}px', (pred_j[0]+5, pred_j[1]-5), fontsize=8, color='gray')

ax.scatter([], [], c='forestgreen', s=60, label='Observed (true)')
ax.scatter([], [], c='tomato', s=40, marker='x', label='Predicted (current estimate)')
ax.set_xlim(0, 640); ax.set_ylim(480, 0)
ax.set_xlabel('u (px)', fontsize=12); ax.set_ylabel('v (px)', fontsize=12)
ax.set_title('Reprojection errors: arrows show the residuals', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

**Key insight:** The goal of bundle adjustment is to find camera poses $\{T_i\}$
and 3D points $\{\mathbf{p}_j\}$ that minimize the sum of squared reprojection
errors across all observations:

$$\min_{T_i, \mathbf{p}_j} \sum_{i,j} \|\mathbf{u}_{ij} - \pi(T_i, \mathbf{p}_j)\|^2$$

## 38.2 Joint Optimization (Gauss Newton BA)

We now implement full bundle adjustment. The state vector contains:
- **5 camera poses**, each parameterized as $(x, y, \theta)$ (3 DOF each)
- **20 landmark positions**, each $(l_x, l_y)$ (2 DOF each)

Total state dimension: $5 \times 3 + 20 \times 2 = 55$.
Observations: bearing angles from cameras to visible landmarks.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_cameras = 5                    # number of cameras
n_landmarks = 20                 # number of landmarks
obs_noise_ba = 0.01              # observation noise (radians)
pose_noise_ba = 0.3              # initial pose perturbation
landmark_noise_ba = 0.5          # initial landmark perturbation
n_ba_iters = 10                  # Gauss-Newton iterations
# ─────────────────────────────────────────────────────────────────────────────

# True cameras: arranged in a semicircle
true_cam_poses = []
for i in range(n_cameras):
    angle = np.pi * i / (n_cameras - 1) - np.pi / 2
    x = 5 * np.cos(angle)
    y = 5 * np.sin(angle)
    theta = angle + np.pi  # face toward center
    true_cam_poses.append(np.array([x, y, theta]))
true_cam_poses = np.array(true_cam_poses)

# True landmarks: scattered in front of cameras
true_landmarks = np.random.uniform(-3, 3, (n_landmarks, 2))

# Generate observations: each camera observes landmarks in its FOV
observations = []  # list of (cam_idx, lm_idx, bearing)
for i in range(n_cameras):
    for j in range(n_landmarks):
        bearing = project_bearing(true_cam_poses[i], true_landmarks[j])
        # Check if in FOV
        if abs(bearing) < np.pi / 3:
            noisy_bearing = bearing + np.random.normal(0, obs_noise_ba)
            observations.append((i, j, noisy_bearing))

print(f'Cameras: {n_cameras}, Landmarks: {n_landmarks}')
print(f'Total observations: {len(observations)}')
print(f'Avg observations per camera: {len(observations)/n_cameras:.1f}')

In [ ]:
# Initialize with noisy estimates
est_cam_poses = true_cam_poses.copy() + np.random.normal(0, pose_noise_ba, true_cam_poses.shape)
# Fix first camera to remove gauge freedom
est_cam_poses[0] = true_cam_poses[0].copy()
est_landmarks = true_landmarks.copy() + np.random.normal(0, landmark_noise_ba, true_landmarks.shape)

def ba_compute_residuals_and_jacobian(cam_poses, landmarks, observations):
    """Compute residuals and Jacobian for bundle adjustment."""
    n_cams = len(cam_poses)
    n_lms = len(landmarks)
    n_obs = len(observations)
    cam_dim = 3  # x, y, theta per camera
    lm_dim = 2   # lx, ly per landmark
    state_dim = n_cams * cam_dim + n_lms * lm_dim
    
    residuals = np.zeros(n_obs)
    J = np.zeros((n_obs, state_dim))
    
    for k, (i, j, z) in enumerate(observations):
        x, y, theta = cam_poses[i]
        lm = landmarks[j]
        R = rot2(-theta)
        t = np.array([x, y])
        local = R @ (lm - t)
        lx, ly = local
        predicted = np.arctan2(ly, lx)
        
        r = predicted - z
        r = (r + np.pi) % (2 * np.pi) - np.pi
        residuals[k] = r
        
        denom = lx**2 + ly**2
        da_dlx = -ly / denom
        da_dly = lx / denom
        
        c, s = np.cos(theta), np.sin(theta)
        dx_lm = lm - t
        
        # Jacobian w.r.t. camera pose (x, y, theta)
        dlx_dx = -c;  dly_dx = s
        dlx_dy = s;   dly_dy = -c
        dlx_dth = s * dx_lm[0] + c * dx_lm[1]
        dly_dth = -c * dx_lm[0] + s * dx_lm[1]
        
        cam_start = i * cam_dim
        if i > 0:  # first camera is fixed
            J[k, cam_start]     = da_dlx * dlx_dx + da_dly * dly_dx
            J[k, cam_start + 1] = da_dlx * dlx_dy + da_dly * dly_dy
            J[k, cam_start + 2] = da_dlx * dlx_dth + da_dly * dly_dth
        
        # Jacobian w.r.t. landmark (lx, ly)
        dlx_dlmx = c;   dly_dlmx = -s
        dlx_dlmy = -(-s); dly_dlmy = c
        
        lm_start = n_cams * cam_dim + j * lm_dim
        J[k, lm_start]     = da_dlx * dlx_dlmx + da_dly * dly_dlmx
        J[k, lm_start + 1] = da_dlx * dlx_dlmy + da_dly * dly_dlmy
    
    return residuals, J

# Run Gauss-Newton
costs_ba = []
for iteration in range(n_ba_iters):
    r, J = ba_compute_residuals_and_jacobian(est_cam_poses, est_landmarks, observations)
    cost = 0.5 * np.sum(r**2)
    costs_ba.append(cost)
    
    # Gauss-Newton: (J^T J + lambda I) dx = -J^T r
    JtJ = J.T @ J
    Jtr = J.T @ r
    dx = np.linalg.solve(JtJ + 1e-6 * np.eye(JtJ.shape[0]), -Jtr)
    
    # Update state
    cam_dim = 3
    for i in range(1, n_cameras):  # skip first camera (fixed)
        est_cam_poses[i] += dx[i*cam_dim:(i+1)*cam_dim]
    lm_start = n_cameras * cam_dim
    for j in range(n_landmarks):
        est_landmarks[j] += dx[lm_start + j*2 : lm_start + j*2 + 2]

print(f'\nBA converged in {n_ba_iters} iterations')
print(f'Initial cost: {costs_ba[0]:.4f}')
print(f'Final cost:   {costs_ba[-1]:.6f}')
print(f'Cost reduction: {costs_ba[0]/costs_ba[-1]:.0f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost convergence
ax = axes[0]
ax.semilogy(costs_ba, 'steelblue', linewidth=2, marker='o', markersize=5)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cost (log scale)', fontsize=12)
ax.set_title('BA cost decrease per iteration', fontsize=13)

# Map comparison
ax = axes[1]
ax.scatter(true_landmarks[:, 0], true_landmarks[:, 1], c='forestgreen', s=40,
           label='True landmarks', edgecolors='k', linewidth=0.5, zorder=5)
ax.scatter(est_landmarks[:, 0], est_landmarks[:, 1], c='steelblue', s=40,
           label='Optimized landmarks', marker='s', zorder=4)

# Draw cameras
for i in range(n_cameras):
    tp = true_cam_poses[i]
    ep = est_cam_poses[i]
    ax.scatter(tp[0], tp[1], c='tomato', s=100, marker='^', zorder=6)
    ax.scatter(ep[0], ep[1], c='orange', s=80, marker='v', zorder=6)

ax.scatter([], [], c='tomato', s=100, marker='^', label='True cameras')
ax.scatter([], [], c='orange', s=80, marker='v', label='Optimized cameras')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('BA result: cameras and landmarks', fontsize=13)
ax.legend(fontsize=9); ax.set_aspect('equal')

plt.tight_layout()
plt.show()

# Errors
cam_errors = np.linalg.norm(est_cam_poses[1:, :2] - true_cam_poses[1:, :2], axis=1)
lm_errors = np.linalg.norm(est_landmarks - true_landmarks, axis=1)
print(f'Camera position RMSE: {np.sqrt(np.mean(cam_errors**2)):.4f} m')
print(f'Landmark RMSE:        {np.sqrt(np.mean(lm_errors**2)):.4f} m')

## 38.3 Local vs Global BA

**Local BA** optimizes only the most recent cameras and the points they observe.
This is fast but does not improve old parts of the map.

**Global BA** optimizes everything. This is slow but produces the best result.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_cams_lg = 8                    # cameras for local/global comparison
n_lms_lg = 30                    # landmarks
local_window = 3                 # number of recent cameras in local BA
# ─────────────────────────────────────────────────────────────────────────────

# Generate cameras on a line
true_cams_lg = np.array([[i * 1.5, 0, 0.0] for i in range(n_cams_lg)])
true_lms_lg = np.random.uniform(-2, 12, (n_lms_lg, 2))
true_lms_lg[:, 1] = np.abs(true_lms_lg[:, 1]) + 2  # ensure in front

# Observations
obs_lg = []
for i in range(n_cams_lg):
    for j in range(n_lms_lg):
        b = project_bearing(true_cams_lg[i], true_lms_lg[j])
        if abs(b) < np.pi / 3:
            obs_lg.append((i, j, b + np.random.normal(0, 0.01)))

# Initial noisy estimates
est_cams_global = true_cams_lg.copy() + np.random.normal(0, 0.3, true_cams_lg.shape)
est_cams_global[0] = true_cams_lg[0].copy()
est_lms_global = true_lms_lg.copy() + np.random.normal(0, 0.4, true_lms_lg.shape)

est_cams_local = est_cams_global.copy()
est_lms_local = est_lms_global.copy()

def run_ba(cam_poses, landmarks, observations, n_iters=5, fixed_cams=None):
    """Run BA, optionally fixing certain cameras."""
    if fixed_cams is None:
        fixed_cams = {0}
    cams = cam_poses.copy()
    lms = landmarks.copy()
    costs = []
    for _ in range(n_iters):
        r, J = ba_compute_residuals_and_jacobian(cams, lms, observations)
        # Zero out Jacobian columns for fixed cameras
        for fc in fixed_cams:
            J[:, fc*3:(fc+1)*3] = 0
        cost = 0.5 * np.sum(r**2)
        costs.append(cost)
        JtJ = J.T @ J
        dx = np.linalg.solve(JtJ + 1e-6 * np.eye(JtJ.shape[0]), -J.T @ r)
        for i in range(len(cams)):
            if i not in fixed_cams:
                cams[i] += dx[i*3:(i+1)*3]
        lm_start = len(cams) * 3
        for j in range(len(lms)):
            lms[j] += dx[lm_start + j*2 : lm_start + j*2 + 2]
    return cams, lms, costs

# Global BA: optimize everything
t0 = time.time()
est_cams_global, est_lms_global, costs_global = run_ba(
    est_cams_global, est_lms_global, obs_lg, n_iters=8)
time_global = time.time() - t0

# Local BA: only optimize last 3 cameras + their visible landmarks
fixed_local = set(range(n_cams_lg - local_window))
local_obs = [(i, j, z) for i, j, z in obs_lg if i >= n_cams_lg - local_window]
t0 = time.time()
est_cams_local, est_lms_local, costs_local = run_ba(
    est_cams_local, est_lms_local, obs_lg, n_iters=8, fixed_cams=fixed_local)
time_local = time.time() - t0

print(f'Global BA: {time_global*1000:.1f} ms, final cost = {costs_global[-1]:.6f}')
print(f'Local BA:  {time_local*1000:.1f} ms, final cost = {costs_local[-1]:.6f}')

In [ ]:
# Compare errors per camera
cam_err_global = np.linalg.norm(est_cams_global[:, :2] - true_cams_lg[:, :2], axis=1)
cam_err_local = np.linalg.norm(est_cams_local[:, :2] - true_cams_lg[:, :2], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x_idx = np.arange(n_cams_lg)
w = 0.35
ax.bar(x_idx - w/2, cam_err_global, w, color='steelblue', alpha=0.8, label='Global BA')
ax.bar(x_idx + w/2, cam_err_local, w, color='tomato', alpha=0.8, label='Local BA')
ax.axvline(n_cams_lg - local_window - 0.5, color='gray', linestyle=':', linewidth=2,
           label=f'Local window boundary')
ax.set_xlabel('Camera index', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('Camera errors: Global vs Local BA', fontsize=13)
ax.legend(fontsize=10)

ax = axes[1]
ax.semilogy(costs_global, 'steelblue', linewidth=2, marker='o', markersize=4, label='Global BA')
ax.semilogy(costs_local, 'tomato', linewidth=2, marker='s', markersize=4, label='Local BA')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cost', fontsize=12)
ax.set_title('Convergence comparison', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'Global BA camera RMSE: {np.sqrt(np.mean(cam_err_global**2)):.4f} m')
print(f'Local BA camera RMSE:  {np.sqrt(np.mean(cam_err_local**2)):.4f} m')

**Key tradeoff:** Local BA is faster but leaves old cameras unoptimized.
Global BA is slower but improves the entire map. In practice, ORB SLAM runs
local BA at keyframe rate and global BA only after loop closures.

## 38.4 Schur Complement

The BA Jacobian has a special **block structure**: camera variables and point
variables are never directly connected to each other (only through observations).
The normal equations $H \Delta x = -g$ have the form:

$$\begin{bmatrix} H_{cc} & H_{cp} \\ H_{pc} & H_{pp} \end{bmatrix}
\begin{bmatrix} \Delta c \\ \Delta p \end{bmatrix} =
\begin{bmatrix} -g_c \\ -g_p \end{bmatrix}$$

The **Schur complement** eliminates the point variables first:

$$(H_{cc} - H_{cp} H_{pp}^{-1} H_{pc}) \Delta c = -g_c + H_{cp} H_{pp}^{-1} g_p$$

This **reduced camera system** is much smaller and faster to solve.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_cams_sc = 5
n_lms_sc = 20
# ─────────────────────────────────────────────────────────────────────────────

# Reuse the BA setup from section 38.2
r_sc, J_sc = ba_compute_residuals_and_jacobian(est_cam_poses, est_landmarks, observations)
H_full = J_sc.T @ J_sc
g_full = J_sc.T @ r_sc

cam_dim_sc = n_cams_sc * 3
lm_dim_sc = n_lms_sc * 2
total_dim = cam_dim_sc + lm_dim_sc

# Extract blocks
H_cc = H_full[:cam_dim_sc, :cam_dim_sc]
H_cp = H_full[:cam_dim_sc, cam_dim_sc:]
H_pc = H_full[cam_dim_sc:, :cam_dim_sc]
H_pp = H_full[cam_dim_sc:, cam_dim_sc:]
g_c = g_full[:cam_dim_sc]
g_p = g_full[cam_dim_sc:]

# Full solve
t0 = time.time()
dx_full = np.linalg.solve(H_full + 1e-6 * np.eye(total_dim), -g_full)
time_full = time.time() - t0

# Schur complement solve
t0 = time.time()
H_pp_inv = np.linalg.inv(H_pp + 1e-6 * np.eye(lm_dim_sc))
S = H_cc - H_cp @ H_pp_inv @ H_pc  # reduced camera system
rhs = -g_c + H_cp @ H_pp_inv @ g_p
dx_cams = np.linalg.solve(S + 1e-6 * np.eye(cam_dim_sc), rhs)
# Back-substitute for points
dx_pts = H_pp_inv @ (-g_p - H_pc @ dx_cams)
dx_schur = np.concatenate([dx_cams, dx_pts])
time_schur = time.time() - t0

print(f'Full system size: {total_dim} x {total_dim}')
print(f'Reduced system size: {cam_dim_sc} x {cam_dim_sc}')
print(f'\nFull solve time:   {time_full*1000:.2f} ms')
print(f'Schur solve time:  {time_schur*1000:.2f} ms')
print(f'Max difference between solutions: {np.max(np.abs(dx_full - dx_schur)):.2e}')

In [ ]:
# Visualize sparsity patterns
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
ax.imshow(np.abs(H_full) > 1e-10, cmap='Blues', aspect='auto')
ax.axhline(cam_dim_sc - 0.5, color='tomato', linewidth=2)
ax.axvline(cam_dim_sc - 0.5, color='tomato', linewidth=2)
ax.set_title(f'Full H ({total_dim}x{total_dim})', fontsize=12)
ax.set_xlabel('Variable index')

ax = axes[1]
ax.imshow(np.abs(S) > 1e-10, cmap='Blues', aspect='auto')
ax.set_title(f'Schur complement S ({cam_dim_sc}x{cam_dim_sc})', fontsize=12)
ax.set_xlabel('Camera variable index')

ax = axes[2]
ax.imshow(np.abs(J_sc) > 1e-10, cmap='Blues', aspect='auto')
ax.axvline(cam_dim_sc - 0.5, color='tomato', linewidth=2)
ax.set_title(f'Jacobian ({J_sc.shape[0]}x{J_sc.shape[1]})', fontsize=12)
ax.set_xlabel('Variable index'); ax.set_ylabel('Observation index')

plt.suptitle('BA sparsity structure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Why Schur complement matters:** In real BA problems with 1000 cameras and
100,000 points, the full system is 303,000 x 303,000. The Schur complement
reduces this to 3,000 x 3,000. That is a 100x reduction, and since solving
scales as $O(n^3)$, it means a **million times** speedup.

## Capstone: Full BA on a Simulated Reconstruction

8 cameras viewing 40 3D points. Add noise to initial estimates.
Run Gauss Newton BA. Show cost decrease, 3D reconstruction improvement,
and camera pose errors before and after.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(7)
n_cap_cams = 8
n_cap_lms = 40
cap_pose_noise = 0.4
cap_lm_noise = 0.6
cap_obs_noise = 0.008
cap_ba_iters = 15
# ─────────────────────────────────────────────────────────────────────────────

# Cameras on a circle
true_cap_cams = np.zeros((n_cap_cams, 3))
for i in range(n_cap_cams):
    angle = 2 * np.pi * i / n_cap_cams
    true_cap_cams[i] = [6*np.cos(angle), 6*np.sin(angle), angle + np.pi]

# Landmarks in center region
true_cap_lms = np.random.uniform(-3, 3, (n_cap_lms, 2))

# Generate observations
cap_obs = []
for i in range(n_cap_cams):
    for j in range(n_cap_lms):
        b = project_bearing(true_cap_cams[i], true_cap_lms[j])
        if abs(b) < np.pi / 3:
            cap_obs.append((i, j, b + np.random.normal(0, cap_obs_noise)))

# Initial estimates with noise
init_cams = true_cap_cams.copy() + np.random.normal(0, cap_pose_noise, true_cap_cams.shape)
init_cams[0] = true_cap_cams[0].copy()  # fix first camera
init_lms = true_cap_lms.copy() + np.random.normal(0, cap_lm_noise, true_cap_lms.shape)

# Record initial errors
init_cam_err = np.linalg.norm(init_cams[1:, :2] - true_cap_cams[1:, :2], axis=1)
init_lm_err = np.linalg.norm(init_lms - true_cap_lms, axis=1)

# Run BA
est_cap_cams = init_cams.copy()
est_cap_lms = init_lms.copy()
cap_costs = []

for iteration in range(cap_ba_iters):
    r, J = ba_compute_residuals_and_jacobian(est_cap_cams, est_cap_lms, cap_obs)
    cost = 0.5 * np.sum(r**2)
    cap_costs.append(cost)
    JtJ = J.T @ J
    dx = np.linalg.solve(JtJ + 1e-6 * np.eye(JtJ.shape[0]), -J.T @ r)
    for i in range(1, n_cap_cams):
        est_cap_cams[i] += dx[i*3:(i+1)*3]
    lm_off = n_cap_cams * 3
    for j in range(n_cap_lms):
        est_cap_lms[j] += dx[lm_off + j*2 : lm_off + j*2 + 2]

final_cam_err = np.linalg.norm(est_cap_cams[1:, :2] - true_cap_cams[1:, :2], axis=1)
final_lm_err = np.linalg.norm(est_cap_lms - true_cap_lms, axis=1)

print(f'Observations: {len(cap_obs)}')
print(f'Initial cost: {cap_costs[0]:.4f}')
print(f'Final cost:   {cap_costs[-1]:.6f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cost convergence
ax = axes[0, 0]
ax.semilogy(cap_costs, 'steelblue', linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('Cost (log)', fontsize=11)
ax.set_title('(a) Cost function decrease', fontsize=12)

# Map: initial vs final vs true
ax = axes[0, 1]
ax.scatter(true_cap_lms[:, 0], true_cap_lms[:, 1], c='forestgreen', s=30,
           label='True', zorder=5, edgecolors='k', linewidth=0.3)
ax.scatter(init_lms[:, 0], init_lms[:, 1], c='tomato', s=20, alpha=0.5,
           label='Initial (noisy)', marker='x')
ax.scatter(est_cap_lms[:, 0], est_cap_lms[:, 1], c='steelblue', s=25,
           label='After BA', marker='s', alpha=0.7)
for i in range(n_cap_cams):
    ax.scatter(true_cap_cams[i, 0], true_cap_cams[i, 1], c='orange', s=80, marker='^', zorder=6)
ax.set_title('(b) 3D reconstruction', fontsize=12)
ax.legend(fontsize=8); ax.set_aspect('equal')

# Camera pose errors
ax = axes[1, 0]
x_idx = np.arange(1, n_cap_cams)
w = 0.35
ax.bar(x_idx - w/2, init_cam_err, w, color='tomato', alpha=0.7, label='Before BA')
ax.bar(x_idx + w/2, final_cam_err, w, color='steelblue', alpha=0.7, label='After BA')
ax.set_xlabel('Camera index', fontsize=11)
ax.set_ylabel('Position error (m)', fontsize=11)
ax.set_title('(c) Camera pose errors', fontsize=12)
ax.legend(fontsize=10)

# Landmark errors histogram
ax = axes[1, 1]
ax.hist(init_lm_err, bins=15, color='tomato', alpha=0.5, label='Before BA')
ax.hist(final_lm_err, bins=15, color='steelblue', alpha=0.5, label='After BA')
ax.set_xlabel('Landmark error (m)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('(d) Landmark error distribution', fontsize=12)
ax.legend(fontsize=10)

plt.suptitle('Capstone: Full Bundle Adjustment Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n=== CAPSTONE RESULTS ===')
print(f'Camera RMSE before BA: {np.sqrt(np.mean(init_cam_err**2)):.4f} m')
print(f'Camera RMSE after BA:  {np.sqrt(np.mean(final_cam_err**2)):.4f} m')
print(f'Landmark RMSE before:  {np.sqrt(np.mean(init_lm_err**2)):.4f} m')
print(f'Landmark RMSE after:   {np.sqrt(np.mean(final_lm_err**2)):.4f} m')
print(f'Cost reduction:        {cap_costs[0]/cap_costs[-1]:.0f}x')

**Capstone takeaways:**
- BA dramatically reduces both camera and landmark errors
- The cost function decreases rapidly in the first few iterations
- Landmark errors are reduced from ~0.5m to sub-centimeter level
- The Schur complement makes this tractable for large problems
- In practice, BA runs after every keyframe insertion (local) and after loop closures (global)

---

## Exercises

### Exercise 38.1
Add **Huber robust loss** to the BA implementation. Inject 5 outlier observations
(bearing error > 0.5 rad). Compare the reconstruction with and without robust loss.

In [ ]:
# Your code here

### Exercise 38.2
Implement **Levenberg Marquardt** instead of Gauss Newton. Start with $\lambda = 1$.
If the cost decreases, divide $\lambda$ by 10. If it increases, multiply by 10 and reject
the step. Compare convergence speed with plain Gauss Newton.

In [ ]:
# Your code here

### Exercise 38.3
Measure the **Schur complement speedup** for BA problems of increasing size.
Generate problems with 5, 10, 20, 50 cameras and proportionally more landmarks.
Plot solve time for full vs Schur complement.

In [ ]:
# Your code here

### Exercise 38.4
Implement **covariance recovery** after BA. The covariance of the state is
$(J^T J)^{-1}$. Extract the marginal covariance of each camera pose and plot
uncertainty ellipses. Which cameras are most uncertain?

In [ ]:
# Your code here

### Exercise 38.5
Compare local BA with windows of size 2, 3, 5, and 8 cameras. For each window size,
run BA 5 times and plot the resulting landmark RMSE vs computation time. Find the
sweet spot that balances accuracy and speed.

In [ ]:
# Your code here